https://read.amazon.com/?asin=B0BVJRKS54&ref_=dbs_t_r_khbodl

In [1]:
5 + 5

10

If, on the other hand, you have no particular preference, then my recommendation is that you start with SQLite, which is by far the easiest to set up and manage. Since the code uses common features present in all the database systems, you can switch to a different database when and if needed. 

# Database Connection URLs

When using SQLAlchemy, a database to connect to is represented by a URL that has the following structure:

```bash
{dialect}{+driver}://{username}:{password}@{hostname}:{port}/{database}
```

Database URLs for SQLite are a bit different, because this is an in-process database without the sconcept of users or servers. For this database, the dialect name is sqlite and the driver can be omitted.

The username, password, hostname and port are also omitted, since they do not have any meaning for this database. Finally, instead of a database name, a path to the datbase file is given.

The following examples show some possible database URLs for a SQLite database named retrofun.sqlite:

```
# database file in the current directory
 url = 'sqlite:///retrofun.sqlite' 
 # database file in /home/miguel/retrofun directory
  url = 'sqlite:////home/miguel/retrofun/retrofun.sqlite'
   # database file in C:\users\miguel\retrofun directory (Microsoft Windows)
 url = 'sqlite:///c:\\users\\miguel\\retrofun\\retrofun.sqlite'
```

If you look at these URLS carefully, you may think that they have too many forward slashes right after the sqlite: prefix, but these are all correct.



In [5]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL", "sqlite:///default.db")
engine = create_engine(os.environ["DATABASE_URL"])

In [6]:
print(DATABASE_URL)

sqlite:///retrofun.sqlite


1. Current Working Directory (CWD)

  When you run a Jupyter notebook, the current working directory is
  typically where you launched Jupyter from (the project root), not where
  the notebook file is located. So even though your notebook is in
  chapter1/, Python's CWD is likely /home/davidd/2026/retrofun/.

  You can verify this by adding a cell:
  import os
  print(os.getcwd())

  2. load_dotenv() Search Behavior

  Even if the CWD were different, load_dotenv() has a built-in search
  feature. When called without arguments, it:
  1. Starts from the current working directory
  2. Searches upward through parent directories
  3. Loads the first .env file it finds

  From the https://saurabh-kumar.com/python-dotenv/:

  load_dotenv() will walk up the directory tree looking for a .env file.

  So if your CWD were chapter1/, it would look in:
  - chapter1/.env (not found)
  - .env (found, loaded)

  This "search upward" behavior is intentional - it lets you run scripts
  from any subdirectory while keeping a single .env file at the project
  root.

In [7]:
import os
os.getcwd()

'/home/davidd/2026/retrofun/chapter1'

In [8]:
print(engine)

Engine(sqlite:///retrofun.sqlite)


# Models

When using the ORM module, database tables are defined in the application as Python classes. The application must create a parent class for all these classes, where settings that are common to all the tables can be configured. This parent class, which SQLAlchemy calls the _declarative base_ class, is often named Model, or in some cases Base. The collection of subclasses of the Model class represent the structure or schema of the database, and are generally referred to as the "models" of the application.


The Model class must inherit from SQLAlchemy's DeclarativeBase class. Here is an updated version of db.py that defines Model as an empy class, without any custom settings.

Model subclasses are configured using class attributes. The __tablename__ attribute defines the name of the database table the class represents. A very common naming convention for database tables is to use the plural form of the entity in lowercase, so in this case the table is given the products name. This contrasts with the convention used for the model class names, which prefers the singular form in camel case.

The remaining attributes defined in the class represent the columns of the table. The Mapped[t] type declaration is used to define each column, with t being the Python type assigned to the column, such as int, str, or datetime. For simple columns such as year above, this is all that is necessary. If the column needs to be given additional options, it is assigned to a mapped_column() constructor that provides those options.

In [9]:
from models import Product

In [36]:
c64 = Product(name="Commodore 64", manufacturer="Commodore", year=1982,
              country="Thailand",cpu="MOS 6510")

In [37]:
c64

Product(None, "Commodore 64")

Even though this object is a model instance, at this point it is just a plain Python object that is not stored or associated with any database.

Note: the concept of model classes is available only for applications that use the ORM module. When using Core, instsances of the Table class are used to represent datbase tables.

In [38]:
c64.metadata

MetaData()

In [39]:
dir(c64.metadata)

['__annotations__',
 '__class__',
 '__class_getitem__',
 '__contains__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__slots__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__visit_name__',
 '__weakref__',
 '_add_table',
 '_compiler_dispatch',
 '_fk_memos',
 '_generate_compiler_dispatch',
 '_init_items',
 '_original_compiler_dispatch',
 '_remove_table',
 '_schema_item_copy',
 '_schemas',
 '_sequences',
 '_set_parent',
 '_set_parent_with_dispatch',
 '_use_schema_map',
 'clear',
 'create_all',
 'create_drop_stringify_dialect',
 'dispatch',
 'drop_all',
 'info',
 'naming_convention',
 'reflect',
 'remove',
 'schema',
 'sorted_tables',
 'tables']

In [40]:
c64.metadata.tables

FacadeDict({'products': Table('products', MetaData(), Column('id', Integer(), table=<products>, primary_key=True, nullable=False), Column('name', String(length=64), table=<products>, nullable=False), Column('manufacturer', String(length=64), table=<products>, nullable=False), Column('year', Integer(), table=<products>, nullable=False), Column('country', String(length=32), table=<products>, nullable=False), Column('cpu', String(length=32), table=<products>, nullable=False), schema=None)})

Here the create_all() method will issue SQL statements to the database represented by engine to create the database tables referenced by all the models. Following the code examples from previous sections, this call would create a products table, which is defined by the Product model. An important limitation of create_all() is that it only creates tables that don't already exist in the database, which means that when a model class is changed, this method cannot be used to transfer the change to the corresponding database table.

Unfortunately updating a database in this way is only practical for small tests or while prototyping, because drop_all() not only deletes the tables but also all the data stored in them. You will later learn how to use Alembic to manage updates to the database in a much more effective way through migration scripts.

In [41]:
c64.metadata.create_all(engine)

In [42]:
from sqlalchemy.orm import Session

In [43]:
with Session(engine) as session:
    try:
        session.add(c64)
        session.commit()
    except:
        session.rollback()
        raise
    print(c64)

Product(1, "Commodore 64")
